In [1]:
# conda env 'at3d_pure_k9'

import pathlib
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent

import sys
sys.path.append(str(PROJECT_ROOT))

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import at3d
import netCDF4 as nc
from collections import OrderedDict
from src.utils.util import load_sensor_dict
import yaml
import time


In [2]:
def load_config(filename=PROJECT_ROOT / 'configs' / 'at3d_default_config.yml'):
    with open(filename, 'r') as file:
        config = yaml.safe_load(file)
    required_keys = [
        'obs_file', 'rte_grid_file', 'background_scatterers_file',
        'cloud_volume_mask_file', 'config_file_name', 'surface_wind_speed',
        'a', 'b', 'error_cov_sf', 'restart_iter', 'out_path'
    ]
    for key in required_keys:
        assert key in config, f'{key} not in config'
    
    # Convert paths in the config to pathlib.Path objects
    required_keys.remove('surface_wind_speed')
    required_keys.remove('a')
    required_keys.remove('b')
    required_keys.remove('error_cov_sf')
    if(config['restart_iter'].lower() == 'none'):
        required_keys.remove('restart_iter')
    for key in required_keys:
        config[key] = PROJECT_ROOT / config[key]
    return config


In [3]:

# if(len(sys.argv) > 1):
#     config = load_config(sys.argv[1])
# elif(len(sys.argv) == 1):
#     config = load_config() # load default config
# else:
#     config = None
config = load_config(PROJECT_ROOT / 'configs' / 'at3d_I_config_test.yml')

iter_dir = config['out_path'] / 'iterations'
supp_file = config['out_path'] / 'loss.csv'

# Define output files
iter_dir.mkdir(parents=True, exist_ok=True)
assert not any(iter_dir.iterdir()), f"Iterations already saved in {iter_dir}.\nSpecify a new output location."

# Load rte grid and observations
rte_grid = xr.load_dataset(config['rte_grid_file'])
obs = load_sensor_dict(config['obs_file'])

# add sensor uncertainty model
inverse_covariance = config['error_cov_sf'] * np.diag(np.ones(4))
uncertainty = at3d.uncertainties.Uncertainty(inverse_covariance, 'L2')
obs.add_uncertainty_model('MISR', uncertainty)

# Make forward sensors
forward_sensors = obs.make_forward_sensors()
wavelength = forward_sensors.get_unique_solvers()[0]

# Define surface reflectance model based on surface wind speed
surfaces = OrderedDict()
m_water = 1.33197 - 1j*2.41e-8
surfaces[wavelength] = at3d.surface.wave_fresnel(np.real(m_water), np.imag(m_water), config['surface_wind_speed'])

# Load SHODM numerical parameters
numerical_parameters = OrderedDict()
numerical_parameters[wavelength] = at3d.configuration.get_config(config['config_file_name'])

# Set solar source
sources = OrderedDict()
S0 = 1361 # W m^-2
r0 = 1 # AU
r = 1.00112201638005
E0 = (r0 / r)**2
F0 = E0 * S0
sources[wavelength] = at3d.source.solar(
    wavelength,
    -1*obs['MISR']['sensor_list'][0].mu0,
    obs['MISR']['sensor_list'][0].SAA,
    solarflux=F0,
    skyrad=0.0
)

# Load background atmosphere
background_optical_scatterers = OrderedDict()
background_properties = xr.load_dataset(config['background_scatterers_file'])
background_optical_scatterers[wavelength] = {}
background_optical_scatterers[wavelength]['background'] = background_properties

# Generate cloud optical properties
mie_mono_table = at3d.mie.get_mono_table(
    'Water',
    (wavelength, wavelength),
    max_integration_radius=65.0,
    minimum_effective_radius=5.5,
    relative_dir=PROJECT_ROOT / 'mie_tables',
    verbose=False
)
mie_mono_tables = OrderedDict()
mie_mono_tables[wavelength] = mie_mono_table
# mie_mono_table.to_netcdf(PROJECT_ROOT / 'mie_tables' / 'red_mie_table.nc')
cloud_optical_prop_gen = at3d.medium.OpticalPropertyGenerator(
    'cloud',
    mie_mono_tables,
    at3d.size_distribution.gamma,
    particle_density=1.0,
    maxnphase=None,
    interpolation_mode='exact',
    density_normalization='density',
    reff=np.linspace(1, 30, 20),
    veff=np.linspace(0.01, 0.2, 5)
)
cloud = rte_grid.copy(deep=True)
cloud['density'] = (['x','y','z'], np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z'])))
reff_profile = config['a'] * rte_grid.z.data + config['b']
reff = np.tile(reff_profile, (rte_grid.dims['x'], rte_grid.dims['y'], 1))
cloud['reff'] = (['x','y','z'], reff)
cloud['veff'] = (['x','y','z'], np.ones(cloud.density.shape) * 0.1)
cloud_optical_properties = cloud_optical_prop_gen(cloud)

# Generate unknown scatterers
grid_to_optical_properties = at3d.medium.GridToOpticalProperties(
    rte_grid,
    'cloud',
    wavelength,
    fixed_dataset=cloud_optical_properties[wavelength].drop_vars('extinction')
)
cloud_volume_mask = xr.load_dataset(config['cloud_volume_mask_file'])['cloud_volume_mask'].data
extinction_coordinate_transform = None
extinction_state2grid_transform = at3d.transforms.StateToGridMask(mask=cloud_volume_mask)
extinction_transforms = (extinction_coordinate_transform, extinction_state2grid_transform)
cloud_unknown_scatterer = at3d.medium.UnknownScatterer(grid_to_optical_properties, extinction=extinction_transforms)
unknown_scatterers = at3d.containers.UnknownScatterers(cloud_unknown_scatterer)

# Set polarization
num_stokes = OrderedDict()
num_stokes[wavelength] = 1

# Set forward model
solvers = at3d.containers.SolversDict()
state_gen = at3d.medium.StateGenerator(
    solvers,
    unknown_scatterers,
    surfaces,
    numerical_parameters,
    sources,
    background_optical_scatterers,
    num_stokes
)
min_bounds, max_bounds = state_gen.transform_bounds()

# Initialize extinction
if(type(config['restart_iter']) == pathlib.PosixPath):
    restart_sensor, restart_solver, restart_rte_grid = at3d.util.load_forward_model(config['restart_iter'])
    extinction_init = restart_solver[wavelength].medium['cloud'].extinction.data
else:
    extinction_init = cloud_volume_mask.astype(float)*0.1
extinction_init_1d = state_gen._unknown_scatterers['cloud'].variables['extinction'].state_to_grid.inverse_transform(extinction_init)
x0 = state_gen._unknown_scatterers['cloud'].variables['extinction'].coordinate_transform.inverse_transform(extinction_init_1d)
state_gen(x0)


In [4]:

# # Forward simulate initialization
# forward_sensors.get_measurements(solvers)


In [5]:

# Set objective function
parallel_solve_kwargs = {
    'n_jobs': 1,
    'mpi_comm': None, 
    'verbose': True, 
    'maxiter': 100, 
    'init_solution': True
}
objective_function = at3d.optimize.ObjectiveFunction.LevisApproxUncorrelatedL2(
    obs,
    solvers,
    forward_sensors,
    unknown_scatterers,
    state_gen,
    state_gen.project_gradient_to_state,
    parallel_solve_kwargs=parallel_solve_kwargs,
    gradient_kwargs={'cost_function': 'L2', 'exact_single_scatter':True},
    uncertainty_kwargs={'add_noise':False},
    min_bounds=min_bounds, 
    max_bounds=max_bounds
)

# Callback function
def save(optimizer):
    if(optimizer.iteration == None):
        outFile = iter_dir / 'iter{:03d}.nc'.format(0)
    else:
        outFile = iter_dir / 'iter{:03d}.nc'.format(optimizer.iteration)
    at3d.util.save_forward_model(outFile, forward_sensors, solvers)
    with open(supp_file, mode='a') as file:
        file.write('{}, {}, {}\n'.format(optimizer.iteration, optimizer.objective_fn.loss, optimizer.objective_fn._total_obj_fn_time))
    return {'None': None}
callback = at3d.callback.CallbackFn(save)

# Generate supplementary file
with open(supp_file, mode='a') as file:
    file.write('iteration, loss, total_obj_fn_time\n')

# Setup optimizer
options = {'maxiter': 100, 'maxls': 25, 'disp': True, 'gtol': 1e-16, 'ftol': 1e-8}
optimizer = at3d.optimize.Optimizer(objective_function, callback_fn=callback, options=options)


In [6]:
### NOTE: SF = 300, Split = 30
F, J = objective_function(x0)

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -1.039  0.522E+02    63375   256.00  1.000   [Radiance 0.670 micron]
     2  -1.529  0.529E+02    63375   256.00  1.000   [Radiance 0.670 micron]
     3  -1.782  0.142E+02    67730   256.00  1.000   [Radiance 0.670 micron]
     4  -1.962  0.138E+02    67730   256.00  1.000   [Radiance 0.670 micron]
     5  -2.854  0.139E+02    67730   256.00  1.000   [Radiance 0.670 micron]
     6  -3.402  0.139E+02    67730   256.00  1.000   [Radiance 0.670 micron]
     7  -3.864  0.139E+02    67730   256.00  1.000   [Radiance 0.670 micron]
     8  -4.159  0.139E+02    67730   256.00  1.000   [Radiance 0.670 micron]
 ! Iterations:      8     Final Criterion:  0.000069   [Radiance 0.670 micron]


In [7]:
print('F = {}'.format(F))
print('J max = {}'.format(np.max(J)))
print('J min = {}'.format(np.min(J)))
print('J med = {}'.format(np.median(J)))
print('J avg = {}'.format(np.mean(J)))

F = 106883.8080687771
J max = 0.1750876887517869
J min = -14.87052234548214
J med = -1.4738732004175024
J avg = -2.452201888371518


In [8]:

# # Save initial conditions
# save(optimizer)


In [9]:

# # Perform retrieval
# start_time = time.time()
# result = optimizer.minimize(x0)
# end_time = time.time()

# # Save final retrieval
# outFile = iter_dir / 'iter{:03d}.nc'.format(optimizer.iteration+1)
# at3d.util.save_forward_model(outFile, forward_sensors, solvers)

# # Print execution time
# print('Execution time: {} seconds'.format(end_time - start_time))

